# 06 · Gap-fill Güçlendirme

## Nereden geliyoruz
Betül'ün `o2-v4`: klasik detection + Hungarian + **gap-fill** → **LB 0.762** (bizim 0.749'dan +0.013).
Gap-fill: `t`'de biten track + `t+2`'de başlayan track ≤10 µm ise, aradaki `t+1`'e sentetik
node koyup **2 kenarı** kurtarır (GT gerekmez → inference'ta çalışır). Sönük çekirdeğin bir karede
kaçmasıyla kopan track'i onarır — bizim 6bba FN sorununun tam çözümü.

**Kritik:** Betül'ün metriği bizimkini düzeltti — resmi **FP kuralı** (tek uç eşleşse bile FP).
Bizim eski eval FP'leri kaçırıyordu (yerel skor şişikti). Bu defter Betül'ün **doğru** metriğini kullanır.

## Bu deney: gap-fill'i güçlendir
1. **2-kare boşluk** (`t→t+3`): hareket <8µm/kare → 2-kare köprü hâlâ makul, daha çok kopuk track kurtarır
2. **Intensity-kontrol**: sentetik node'u sadece o konumda **gerçekten foreground** varsa koy → FP azalt

Detection bir kez; gap-fill varyantları aynı tespitler üzerinde **doğru metrikle** kıyaslanır.

## 0 · Kurulum (Betül'ün güvenli sys.path.append hattı)

In [ ]:
import sys, os, time, warnings
from pathlib import Path
from collections import Counter
LIBS=Path('/kaggle/input/datasets/erdeemt/cell-tracking-libs/pylibs')
if not LIBS.exists():
    hits=[Path(r) for r,d,f in os.walk('/kaggle/input') if os.path.basename(r)=='pylibs']
    assert hits, 'cell-tracking-libs dataset ekli degil!'; LIBS=hits[0]
sys.path.append(str(LIBS))         # append! ortamin numpy/scipy'sini golgeleme
import numpy as np, pandas as pd, zarr
from scipy import ndimage as ndi
from scipy.spatial.distance import cdist
from scipy.optimize import linear_sum_assignment
from skimage.filters import threshold_otsu
warnings.filterwarnings('ignore')
print('zarr', zarr.__version__, '| numpy', np.__version__)
assert np.__version__.startswith('2.0'), f'numpy golgelendi: {np.__version__}'
print('hazir')

## 1 · Config + veri

In [ ]:
SCALE_ZYX=(1.625,0.40625,0.40625); S=np.array(SCALE_ZYX,np.float32)
LINK_MAX_UM=8.0; MATCH_UM=7.0
SIGMA=(1,2,2); FOOT=(3,11,11)
GAP_MAX_UM=10.0                    # 1-kare bosluk gate (Betul)
INPUT=Path('/kaggle/input'); COMP=INPUT/'competitions'/'biohub-cell-tracking-during-development'
def find_root():
    if (COMP/'train').is_dir() and (COMP/'test').is_dir(): return COMP
    st=[(INPUT,0)]
    while st:
        b,d=st.pop()
        try:
            if (b/'train').is_dir() and (b/'test').is_dir(): return b
        except Exception: pass
        if d<5:
            for c in sorted(b.iterdir()):
                if c.is_dir() and not c.name.endswith(('.zarr','.geff')): st.append((c,d+1))
ROOT=find_root(); TRAIN=ROOT/'train'; TEST=ROOT/'test'
test_names=sorted(p.stem for p in TEST.glob('*.zarr'))
DEV=len(test_names)>0 and (TRAIN/(test_names[0]+'.geff')).exists()
def open_image(zp):
    n=zarr.open(str(zp),mode='r'); a=dict(n.attrs)
    ms=a.get('multiscales') or (a['ome'].get('multiscales') if isinstance(a.get('ome'),dict) else None)
    return n[ms[0]['datasets'][0]['path']] if ms else (n['0'] if '0' in list(n.keys()) else n)
def load_geff(gp):
    g=zarr.open(str(gp),mode='r'); d={'id':np.asarray(g['nodes/ids'])}
    for k in ('t','z','y','x'): d[k]=np.asarray(g[f'nodes/props/{k}/values'])
    return pd.DataFrame(d), np.asarray(g['edges/ids'])
print('ROOT:',ROOT,'| test:',len(test_names),'| MOD:', 'DEV' if DEV else 'RERUN')

## 2 · Detection (Betül'le aynı) + foreground maskeleri

In [ ]:
def detect_frame(v):
    sm=ndi.gaussian_filter(v.astype(np.float32),sigma=SIGMA); thr=threshold_otsu(sm)
    mx=ndi.maximum_filter(sm,size=FOOT); pk=(sm==mx)&(sm>thr)
    lbl,n=ndi.label(pk)
    if n==0: return np.zeros((0,3),np.float32)
    return np.asarray(ndi.center_of_mass(sm,lbl,np.arange(1,n+1)),np.float32)

def detect_and_fg(arr):
    # cents (merkezler) + fg (foreground maskeleri) TEK gecise -> vol_check icin
    cents=[]; fg={}
    for t in range(arr.shape[0]):
        v=np.asarray(arr[t]).astype(np.float32)
        sm=ndi.gaussian_filter(v,sigma=SIGMA); thr=threshold_otsu(sm)
        mask=sm>thr
        mx=ndi.maximum_filter(sm,size=FOOT); pk=(sm==mx)&mask
        lbl,n=ndi.label(pk)
        cents.append(np.asarray(ndi.center_of_mass(sm,lbl,np.arange(1,n+1)),np.float32)
                     if n else np.zeros((0,3),np.float32))
        fg[t]=mask
    return cents, fg
print('ok')

## 3 · Linking + **güçlendirilmiş** gap-fill

In [ ]:
def link_pairs(A,B):
    if len(A)==0 or len(B)==0: return []
    D=cdist(A*S,B*S); cost=np.where(D<=LINK_MAX_UM,D,1e6)
    r,c=linear_sum_assignment(cost)
    return [(int(i),int(j)) for i,j in zip(r,c) if D[i,j]<=LINK_MAX_UM]

def build_graph(cents):
    nodes,edges,off,nid=[],[],[],1
    for t,c in enumerate(cents):
        off.append(nid)
        for p in c: nodes.append((nid,t,int(round(p[0])),int(round(p[1])),int(round(p[2])))); nid+=1
    for t in range(len(cents)-1):
        for i,j in link_pairs(cents[t],cents[t+1]): edges.append((off[t]+i,off[t+1]+j))
    return nodes,edges

def gap_fill(nodes, edges, fg=None, max_gap=1, vol_check=False, gap_um=GAP_MAX_UM):
    # t'de biten (cocuksuz) + t+g+1'de baslayan (ebeveynsiz) track'leri g-kare boslukla kopru.
    # g adet ara node linear-interp. vol_check: ara node foreground'da degilse kopruleme (FP azalt).
    if not nodes: return nodes, edges
    pos={n[0]:np.array([n[2],n[3],n[4]],float) for n in nodes}
    has_child=set(u for u,v in edges); has_parent=set(v for u,v in edges)
    ends={}; starts={}
    for nid,t,z,y,x in nodes:
        if nid not in has_child: ends.setdefault(int(t),[]).append(nid)
        if nid not in has_parent: starts.setdefault(int(t),[]).append(nid)
    nn=list(nodes); ne=list(edges); nxt=max(n[0] for n in nodes)+1
    used_e=set(); used_s=set()
    for g in range(1, max_gap+1):              # once 1-kare, sonra 2-kare bosluk
        gate=gap_um+(g-1)*4.0                   # 2-kare biraz daha uzak (hareket 2-4um/kare)
        for f in sorted(ends):
            e=[x for x in ends.get(f,[]) if x not in used_e]
            s=[x for x in starts.get(f+g+1,[]) if x not in used_s]
            if not e or not s: continue
            A=np.array([pos[i] for i in e]); B=np.array([pos[j] for j in s])
            D=cdist(A*S,B*S); cost=np.where(D<=gate,D,1e9); r,c=linear_sum_assignment(cost)
            for i,j in zip(r,c):
                if D[i,j]>gate: continue
                p0,p1=pos[e[i]],pos[s[j]]; mids=[]; ok=True
                for k in range(1,g+1):
                    a=k/(g+1); m=p0*(1-a)+p1*a; mids.append((f+k,m))
                    if vol_check and fg is not None:
                        zz,yy,xx=int(round(m[0])),int(round(m[1])),int(round(m[2]))
                        M=fg.get(f+k)
                        if M is not None and not M[min(zz,M.shape[0]-1),min(yy,M.shape[1]-1),min(xx,M.shape[2]-1)]:
                            ok=False; break                  # ara konum foreground DEGIL -> kopruleme
                if not ok: continue
                prev=e[i]
                for (ft,m) in mids:
                    sid=nxt; nxt+=1
                    nn.append((sid,ft,int(round(m[0])),int(round(m[1])),int(round(m[2]))))
                    ne.append((prev,sid)); prev=sid
                ne.append((prev,s[j])); used_e.add(e[i]); used_s.add(s[j])
    return nn,ne
print('ok')

## 4 · Resmi metrik (Betül'ün — FP kuralı düzeltilmiş)

In [ ]:
def eval_vs_gt(nodes, edges, gt_ndf, gt_edges):
    pn=pd.DataFrame(nodes,columns=['node_id','t','z','y','x']); gmap={}
    for t,g in gt_ndf.groupby('t'):
        p=pn[pn.t==int(t)]
        if len(p)==0 or len(g)==0: continue
        D=cdist(p[['z','y','x']].values*S, g[['z','y','x']].values*S)
        cost=np.where(D<=MATCH_UM,D,1e6); r,c=linear_sum_assignment(cost)
        pid=p['node_id'].values; gid=g['id'].values
        for i,j in zip(r,c):
            if D[i,j]<=MATCH_UM: gmap[int(pid[i])]=int(gid[j])
    gtset=set(); gt_out={}; gt_in={}
    for u,v in gt_edges:
        u,v=int(u),int(v); gtset.add((u,v))
        gt_out.setdefault(u,set()).add(v); gt_in.setdefault(v,set()).add(u)
    eTP=eFP=0; cov=set()
    for pu,pv in edges:
        gu,gv=gmap.get(pu),gmap.get(pv)
        if gu is not None and gv is not None and (gu,gv) in gtset: eTP+=1; cov.add((gu,gv))
        elif (gu is not None and len(gt_out.get(gu,())) >0) or (gv is not None and len(gt_in.get(gv,())) >0): eFP+=1
    eFN=len(gtset)-len(cov)
    return dict(eTP=eTP,eFP=eFP,eFN=eFN,
                jaccard=round(eTP/max(eTP+eFP+eFN,1),4),
                node_recall=round(len(set(gmap.values()))/max(len(gt_ndf),1),4),
                pred_nodes=len(nodes))

def estimate_true_count(cents):
    T=len(cents); ts=np.linspace(0,T-1,min(6,T)).astype(int)
    return int(np.median([len(cents[int(t)]) for t in ts])*T)

def micro_final(rows):
    # ceza video-basina, w_i=TP+FP+FN agirlikli ort (resmi)
    num=den=0.0
    for r in rows:
        w=r['eTP']+r['eFP']+r['eFN']
        if w==0: continue
        tt=max(r['T_true'],1); pen=max(0.0,1-0.1*(r['pred_nodes']-tt)/tt)
        num+=w*(r['eTP']/w)*pen; den+=w
    eTP=sum(r['eTP'] for r in rows); eFP=sum(r['eFP'] for r in rows); eFN=sum(r['eFN'] for r in rows)
    return dict(adj_edge_J=round(num/max(den,1),4), raw=round(eTP/max(eTP+eFP+eFN,1),4),
                eTP=eTP,eFP=eFP,eFN=eFN)
# sanity
if DEV:
    gn,ge=load_geff(TRAIN/(test_names[0]+'.geff'))
    gt=[(int(r.id),int(r.t),float(r.z),float(r.y),float(r.x)) for r in gn.itertuples()]
    chk=eval_vs_gt(gt,[(int(u),int(v)) for u,v in ge],gn,ge)
    assert chk['jaccard']>0.999, 'SANITY FAIL'; print('GT->GT jaccard=1.0 ✓')

## 5 · Sweep — gap-fill varyantları (doğru metrikle)
Detection **bir kez** (pahalı). Her varyant aynı tespitler üzerinde. `adj_edge_J` = LB yönü.

In [ ]:
VARIANTS=[("gap0 (yok)",       0, False),
          ("gap1 (Betul)",      1, False),
          ("gap1+vol",          1, True),
          ("gap2",              2, False),
          ("gap2+vol",          2, True)]
if DEV:
    CACHE={}
    for nm in test_names:
        t0=time.time(); arr=open_image(TEST/(nm+'.zarr'))
        cents,fg=detect_and_fg(arr); gn,ge=load_geff(TRAIN/(nm+'.geff'))
        CACHE[nm]=(cents,fg,gn,ge,estimate_true_count(cents))
        print(f"  {nm}: detection {time.time()-t0:.0f}s")
    print()
    results={}
    for label,mg,vc in VARIANTS:
        rows=[]
        for nm in test_names:
            cents,fg,gn,ge,tt=CACHE[nm]
            nodes,edges=build_graph(cents)
            nodes,edges=gap_fill(nodes,edges,fg,max_gap=mg,vol_check=vc)
            r=eval_vs_gt(nodes,edges,gn,ge); r['T_true']=tt; rows.append(r)
        m=micro_final(rows); results[label]=m
        print(f"{label:16s} adj_edge_J={m['adj_edge_J']:.4f}  raw={m['raw']:.4f}  "
              f"[eTP={m['eTP']} eFP={m['eFP']} eFN={m['eFN']}]")
    best=max(results, key=lambda k: results[k]['adj_edge_J'])
    print(f"\n>> EN IYI: {best} (adj_edge_J={results[best]['adj_edge_J']})")
    print(">> (Betul o2-v4: gap1 -> adj 0.7014, LB 0.762)")
else:
    print("RERUN -> sweep atlandi")

## 6 · Karar
- [ ] En iyi varyant = **…**, adj_edge_J = **…** (Betül gap1: 0.7014)
- [ ] 2-kare kazandırıyor mu? intensity-kontrol FP'yi düşürüyor mu (eFP)?
- [ ] Kazanan > gap1 ise → `02_baseline`/submit hattına al, submit (Betül 0.762'nin üstü)

**Kaybederse:** gap1 (Betül) zaten 0.762 — orada kal. Sıradaki: **U-Net detection (YOL B)** — asıl kaldıraç.